In [1]:
from PIL import Image
import torch
from torch import nn
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode# original https://github.com/mlfoundations/open_clip (not for this code)

In [2]:
!nvidia-smi

Mon Oct 19 09:21:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 PCIe               On  |   00000000:B5:00.0 Off |                    0 |
| N/A   52C    P0             98W /  350W |    6237MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
import sys, importlib, os
# Adds the 'src' directory to the path so you can import its subfolders directly
sys.path.append(os.path.abspath("src"))
path_to_root = os.path.abspath(os.path.join('..'))
if path_to_root not in sys.path:
    sys.path.append(path_to_root)

In [4]:
import scripts.custom_models.monet_custom_classifer
import importlib
importlib.reload(scripts.custom_models.monet_custom_classifer)
from geom_hyp import lorentz as L
importlib.reload(L)
from geom_hyp import manifold_conversion as MC


from scripts.custom_models.monet_custom_classifer import MonetWithHead, MonetWithHypHead

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [6]:
# Hyperbolic Network
results = torch.load("../Datasets/sarker_embeddingDerm12345 datasets_64_small_radius.pt")
z_p = results["data"]  # torch.Tensor
z_l = MC.poincare_to_lorentz(z_p) # lorentz points

mul_factor = 2.0
z_l = L.exp_map0(L.log_map0(z_l[:, 1:]) * mul_factor)

# 0:9 for the isic, 0:40 for derm, 0:10 for milk10k
model = MonetWithHypHead(64, l_target= z_l[0:40, :].to(device), loss_type= 'ce', logit_scale = 3, entail_weight= 0.1, multilayer= True) # lorentz space components [selecting  the leaf classes]
# model = DermLIPWithHypHead(64, l_target= z_l[0:40,1:].to(device), loss_type= 'ce', logit_scale = 3, entail_weight= 0.02) # lorentz space components [selecting  the leaf classes]

/scratch/26547344/ipykernel_704171/3720909634.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  results = torch.load("../Datasets/sarker_embeddingDerm12345 datasets_64_sma

In [7]:
torch.acosh(model.l_target.pow(2).sum(dim = -1).add(1).sqrt())
torch.acosh(model.l_target.pow(2).sum(dim = -1).add(1).sqrt())
mini = L.pairwise_dist(model.l_target, model.l_target)
x_masked = mini.clone()
x_masked.fill_diagonal_(float("inf"))

min_val = x_masked.min()
idx = torch.argwhere(x_masked == min_val)[0]
print(min_val)
print(idx)  # tensor([row, col])

tensor(0.6224, device='cuda:0')
tensor([21, 38], device='cuda:0')


In [ ]:
# Euclidean Baseline
print('check run:Euclidean Model')
model = MonetWithHead(11)

check run:Euclidean Model


In [7]:
# Augmentation Scheme Loop + data loading. 

import Datasets.MILK_10K.Milk10_dataset_loader 
import Datasets.isic_kaggle_2016.IsicKaggle2016
import Datasets.fitzpatrick17k.fitzpatrick_data_loader

importlib.reload(Datasets.MILK_10K.Milk10_dataset_loader)

from Datasets.MILK_10K.Milk10_dataset_loader import MILK10kDataset, get_image_transform
from Datasets.isic_kaggle_2016.IsicKaggle2016 import TumorDataset
from Datasets.fitzpatrick17k.fitzpatrick_data_loader import Fitz17kDataset

import Datasets.MILK_10K.augmentation_scheme_image
import importlib
importlib.reload(Datasets.MILK_10K.augmentation_scheme_image)

from Datasets.MILK_10K.augmentation_scheme_image import _augmentation_gpu


# data loading. 

import sys, importlib
path_to_root = os.path.abspath(os.path.join('..'))
if path_to_root not in sys.path:
    sys.path.append(path_to_root)




# Derm12345 loading

path_to_root = os.path.abspath(os.path.join('../../../../../projects/academic/sangst/zahidhas/Datasets'))
if path_to_root not in sys.path:
    sys.path.append(path_to_root)

from DERM12345.derm12345_data_loader import DERM12345Dataset


# ISIC loading
from isic_2019.isic_2016_dataset_loader import ISIC2019Dataset

In [8]:
im_size = 224

transform_train_no_np = transforms.Compose([
    transforms.Resize((im_size, im_size),  interpolation=InterpolationMode.BICUBIC),  # Resize the image
    transforms.PILToTensor()  # Convert to uint8 tensor array [c, h, w]
])

constants_zoo = {
    'imagenet': {'mean': (0.485, 0.456, 0.406), 'std': (0.228, 0.224, 0.225)},
    'openai_clip':{'mean': (0.48145466, 0.4578275, 0.40821073), 'std': (0.26862954, 0.26130258, 0.27577711)},
    'uniform': {'mean': (0.5, 0.5, 0.5), 'std': (0.5, 0.5, 0.5)}
}
constants = constants_zoo['openai_clip']

mean, std = constants.get('mean'), constants.get('std')
dl_preprocess = transforms.Compose([
    transforms.Normalize(mean=mean, 
                         std=std),
])

In [8]:
# MILK 10K dataset
dataset_keep_track = "MILK 10K datasets"
print(dataset_keep_track)

root             = '../Datasets/MILK_10K/Train'
split            = 'train'

train_dataset =  MILK10kDataset(
        root             = root, 
        split            = split,
        image_transform  = get_image_transform(split, im_size),
        balanced         = False,
        seed             = 44,
    )


MILK 10K datasets


In [27]:
# ISIC 2019 datasets
root = "../../../../../projects/academic/sangst/zahidhas/Datasets/isic_2019"

dataset_keep_track = "Isic2019 datasets"

img_size = 224

train_dataset = ISIC2019Dataset(
    root=root,
    split="train",
    image_transform=get_image_transform('train', img_size=img_size),
    balanced=False,
    seed=44,
)

In [9]:
# Derm12345 dataset
dataset_keep_track = "Derm12345 datasets"
print(dataset_keep_track)

root             = "../../../../../projects/academic/sangst/zahidhas/Datasets/DERM12345/"
img_size = 224
train_dataset  = DERM12345Dataset(
        root=root,
        split="train",
        image_transform=get_image_transform("train", img_size = img_size),
        target_col="label",
        balanced=False,
        seed=44,)

Derm12345 datasets


In [10]:
from torch.utils.data import DataLoader

print(dataset_keep_track)
train_loader = DataLoader(
    train_dataset,
    batch_size= 96 ,
    shuffle=True,
    num_workers=0, 
    persistent_workers=False,
)

Derm12345 datasets


In [11]:

# for name, value in vars(model_2).items():
#         print(f"{name}: {value}")

model = model.to(device)

# Define parameter groups
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR
import math

lr1 = 8e-5   # lower lr for pretrained ViT backbone
lr2 = 8e-4   # higher lr for classification head

param_groups = []
all_params = list(model.parameters())
added_params = set()

def add_params(mod, lr):
    for p in mod.parameters():
        if p not in added_params:
            param_groups.append({"params": [p], "lr": lr})
            added_params.add(p)
        
# Custom groups
if hasattr(model, "visual"):
    add_params(model.visual, lr1)

if hasattr(model, "clf_head"):
    add_params(model.clf_head, lr2)

if hasattr(model, "projection_head"):
    add_params(model.clf_head, lr2)

# Add any remaining parameters not covered
for p in model.parameters():
    if p not in added_params:
        param_groups.append({"params": [p], "lr": lr1})


# optimizer = optim.AdamW(param_groups, weight_decay=1e-4)  # AdamW is standard for ViT
## or alternatively
optimizer = optim.SGD(param_groups, momentum=0.99, weight_decay=1e-4) # comment the scheduler in this case


# Optional sanity check
covered = sum(p.numel() for g in optimizer.param_groups for p in g["params"])
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Covered parameters: {covered}/{total} ({covered/total:.2%})")

# Optional: list groups and learning rates
# for i, g in enumerate(optimizer.param_groups):
#     print(f"Group {i}: lr={g['lr']}, num_params={len(g['params'])}")

# # Warmup + Cosine Decay Scheduler
# warmup_epochs = 5
# total_epochs = 100

# def lr_lambda(current_epoch):
#     if current_epoch < warmup_epochs:

#         # Linear warmup
#         return current_epoch / max(1, warmup_epochs)
#     else:
#         # Cosine decay after warmup
#         progress = (current_epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
#         return 0.5 * (1.0 + math.cos(math.pi * progress))

# scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)

# # In training loop:
# # optimizer.step()
# # scheduler.step()  # call once per epoch


✅ Covered parameters: 303461187/303458625 (100.00%)


### ⚠️ Note: AMP Can Cause NaN Gradient Norms

When using **Automatic Mixed Precision (AMP)**, it is possible to observe a **finite loss but a `NaN` gradient norm**:

```text
loss      = finite
grad_norm = NaN
```

This can happen because AMP executes selected operations in lower precision (e.g., FP16/BF16). The forward computation may remain numerically valid while the **backward computation becomes unstable**, producing `Inf` or `NaN` gradients.

This is particularly important for numerically sensitive operations such as:

- `acosh`
- `acos`
- `sqrt`
- `exp`
- vector norms
- divisions near numerical/domain boundaries
- hyperbolic distance and exponential-map operations

For example, a function can have a perfectly finite forward value while its derivative becomes extremely large near a boundary.

#### GPU Dependence

AMP numerical behavior can also be **GPU-specific**.

Different GPU architectures may use different mixed-precision kernels and numerical implementations. The behavior can additionally depend on:

- GPU architecture/model
- CUDA version
- cuDNN version
- PyTorch version
- FP16/BF16 support
- kernel selected by PyTorch

Therefore, it is possible that:

```text
GPU A + AMP  → finite gradients
GPU B + AMP  → NaN gradients
```

even when using the **same model, code, data, and hyperparameters**.

#### Recommended Diagnostic

If `grad_norm` becomes `NaN` with AMP, run the same model/batch in **full FP32** by removing `autocast` and `GradScaler`:

```python
optimizer.zero_grad()

outputs = model(
    input_image,
    labels,
    ignore_index=ignore_index,
    mul_factor=mul_factor
)

loss = outputs["loss"]

# Check forward value
if not torch.isfinite(loss):
    print("Non-finite loss:", loss.item())

# Standard FP32 backward
loss.backward()

# Check gradients BEFORE clipping
for name, p in model.named_parameters():
    if p.grad is not None and not torch.isfinite(p.grad).all():
        print("BAD GRAD:", name)

# Gradient clipping is still used
grad_norm = torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    grad_clip
)

print("Loss:", loss.item())
print("Gradient norm:", grad_norm.item())

optimizer.step()
```

If the comparison gives:

```text
FP32 → finite loss + finite gradient norm
AMP  → finite loss + NaN gradient norm
```

then the **AMP/mixed-precision execution path is a strong suspect**.

> **Important:** Gradient clipping cannot repair a gradient that is already `NaN`. Always check gradient finiteness before the optimizer step when debugging this issue.

> **Practical note:** If training works in FP32 but fails with AMP, especially after moving to a different GPU, first test whether the problem is GPU/mixed-precision specific before changing the model architecture or loss functions.

# training Loop
import random
import pdb
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler
im_size = 224
scaler = GradScaler()
grad_clip = 4  # gradient clipping range

ignore_index = -100
# zero_shot = False
# un_l = [3,5,13,19]  # unlabeled class indices
model.train()
total_epochs = 1 # No of Epoch

for epoch in range(total_epochs):  # loop over the dataset multiple times    
    running_loss = 0.0
    # pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{total_epochs}")
    for i, data in enumerate(train_loader, 0):
        # --- Data loading ---
        orig_image, orig_label = data
        
        if dataset_keep_track == "MILK 10K datasets":
            key = random.choice(['dermoscopic', 'clinical_closeup'])
            images = orig_image[key]
            # labels = orig_label['class_vector'].long() # Euclidean
            labels = orig_label['class_index']
            
        elif dataset_keep_track in ["ISIC 2016 datasets", "Fitzpatrick datasets", 'Derm12345 datasets', 'Isic2019 datasets']:
            images = orig_image         # adjust key as needed
            labels = orig_label['class_index'].long()
        else:
            raise ValueError(f"Unknown dataset: {dataset_keep_track}")

        images = images.to(device)
        
        images = _augmentation_gpu(
            images.float(),
            augmentation_num=random.randint(0, 2),
            im_size=im_size
        )
        images = images/255.0
        input_image = dl_preprocess(images)  # if model is fp16
        input_image = input_image.to(device)
        labels = labels.to(device)

        if not (labels != ignore_index).any():continue        
            
        # --- Forward + backward ---
        # --- Forward ---
        optimizer.zero_grad()
        with autocast():
            outputs = model(input_image, labels, ignore_index = ignore_index, mul_factor = mul_factor)
            loss = outputs["loss"]
        # --- NaN check before backward ---
        if not torch.isfinite(loss):
            print(f"\n[NaN Detected] Epoch {epoch+1} Step {i}")
            print("Logging:", outputs.get("logging", {}))
            pdb.set_trace()
        # --- Backward ---#
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer)
        scaler.update()
        # --- Logging --- #
        running_loss += loss.item()
        avg_loss = running_loss / (i + 1)
        # pbar.set_postfix(loss=f"{loss.item():.4f}", avg=f"{avg_loss:.4f}")
        if i % 100 == 0:
            print(f"[Epoch {epoch} | Step {i}] loss: {loss.item():.4f}")

    # scheduler.step() 
    print(f"Epoch {epoch} avg loss: {running_loss / len(train_loader):.4f}")

In [15]:
# Training Loop — AMP completely removed
print("with AMP look earlier notes")
import random
import pdb
from tqdm import tqdm

im_size = 224
grad_clip = 4
ignore_index = -100

model.train()
total_epochs = 110

for epoch in range(total_epochs):
    running_loss = 0.0

    for i, data in enumerate(train_loader, 0):

        # --- Data loading ---
        orig_image, orig_label = data
        if dataset_keep_track == "MILK 10K datasets":
            key = random.choice(['dermoscopic', 'clinical_closeup'])
            images = orig_image[key]
            # labels = orig_label['class_vector'].long() # Euclidean
            labels = orig_label['class_index']
            
        elif dataset_keep_track in ["ISIC 2016 datasets", "Fitzpatrick datasets", 'Derm12345 datasets', 'Isic2019 datasets']:
            images = orig_image         # adjust key as needed
            labels = orig_label['class_index'].long()
        else:
            raise ValueError(f"Unknown dataset: {dataset_keep_track}")

        images = images.to(device)
        
        images = _augmentation_gpu(
            images.float(),
            augmentation_num=random.randint(0, 3),
            im_size=im_size
        )
        images = images/255.0
        input_image = dl_preprocess(images)  # if model is fp16
        input_image = input_image.to(device)
        labels = labels.to(device)
        if not (labels != ignore_index).any():continue        

        # --- Forward ---
        optimizer.zero_grad()

        outputs = model(input_image, labels,ignore_index=ignore_index, mul_factor=mul_factor)
        loss = outputs["loss"]

        # --- Loss check ---
        if not torch.isfinite(loss):
            print(f"\n[Non-finite loss] Epoch {epoch+1} Step {i}")
            pdb.set_trace()

        # --- Backward: standard FP32 ---
        loss.backward()

        # --- Check gradients BEFORE clipping ---
        bad_grad = False

        for name, p in model.named_parameters():
            if p.grad is not None and not torch.isfinite(p.grad).all():
                print("BAD GRAD:", name)
                bad_grad = True

        if bad_grad:
            print(f"\n[Non-finite gradient] Epoch {epoch+1} Step {i}")
            pdb.set_trace()

        # --- Gradient clipping ---
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(),grad_clip)

        # print(
        #     f"loss={loss.item():.6f}, "
        #     f"grad_norm={grad_norm.item():.6f}"
        # )
        # --- Update ---
        optimizer.step()

        # --- Logging ---
        running_loss += loss.item()
        avg_loss = running_loss / (i + 1)

        if i % 100 == 0:
            print(f"[Epoch {epoch} | Step {i}] " f"loss: {loss.item():.4f}")


    print(f"Epoch {epoch} avg loss: " f"{running_loss / len(train_loader):.4f}")

with AMP look earlier notes
[Epoch 0 | Step 0] loss: 0.7392
[Epoch 0 | Step 100] loss: 0.3762
Epoch 0 avg loss: 0.5580
[Epoch 1 | Step 0] loss: 0.6398
[Epoch 1 | Step 100] loss: 0.4064
Epoch 1 avg loss: 0.5860
[Epoch 2 | Step 0] loss: 0.2269
[Epoch 2 | Step 100] loss: 0.4612
Epoch 2 avg loss: 0.5799
[Epoch 3 | Step 0] loss: 0.6021
[Epoch 3 | Step 100] loss: 0.2856
Epoch 3 avg loss: 0.5600
[Epoch 4 | Step 0] loss: 0.3275
[Epoch 4 | Step 100] loss: 0.6189
Epoch 4 avg loss: 0.5376
[Epoch 5 | Step 0] loss: 0.1877
[Epoch 5 | Step 100] loss: 0.3913
Epoch 5 avg loss: 0.5336
[Epoch 6 | Step 0] loss: 0.2253
[Epoch 6 | Step 100] loss: 0.3577
Epoch 6 avg loss: 0.5425
[Epoch 7 | Step 0] loss: 0.3271


KeyboardInterrupt: 

In [ ]:
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

print(grad_norm.item())
for name, p in model.named_parameters():
    if p.grad is not None and not torch.isfinite(p.grad).all():
        print("BAD GRAD:", name,
              "nan:", torch.isnan(p.grad).any().item(),
              "inf:", torch.isinf(p.grad).any().item())

In [33]:
# with torch.no_grad():
#     vit_oupt = model.visual(input_image)
#     model_oupt = model.clf_head(vit_oupt)
#     #vit_oupt.norm(dim = -1).mean()

In [38]:
# some post analysis, may delete it later
dd = L.pairwise_dist(outputs['output'], model.l_target)
model.logit_scale.exp()
data_sample = 35
print(torch.softmax(2*(-dd), dim = -1)[data_sample])
labels[data_sample]

tensor([0.0055, 0.1636, 0.0394, 0.0439, 0.0311, 0.0185, 0.0684, 0.0905, 0.0270,
        0.4508, 0.0613], device='cuda:0', grad_fn=<SelectBackward0>)


tensor(9, device='cuda:0')

In [15]:
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"Device Capability: {torch.cuda.get_device_capability(0)}")

#ls ../../../../../projects/academic/sangst/zahidhas/Model_Weights/Monet/ # weight save directory

PyTorch Version: 2.5.0a0+872d972e41.nv24.08
CUDA Available: True
GPU Model: NVIDIA A16
Device Capability: (8, 6)


In [16]:
# Saving block. 
PATH = '../../../../../projects/academic/sangst/zahidhas/Model_Weights/Monet/Monet_hyper_sarkar_head_64_Derm12345_weight_ce_multilayer_small_radius_2.pth'


response = input(
    f"\nCheckpoint path:\n{PATH}\n\n"
    "Type SAVE to save the model, or ABORT to cancel: ").strip()


if response == "SAVE":
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        # "scheduler_state_dict": scheduler.state_dict(),
        "loss": loss,
    }, PATH)

    print(f"✅ Model saved to {PATH}")

elif response == "ABORT":
    print("🛑 Save aborted.")

else:
    print(f"❌ Invalid response '{response}'. Nothing was saved.")


Checkpoint path:
../../../../../projects/academic/sangst/zahidhas/Model_Weights/Monet/Monet_hyper_sarkar_head_64_Derm12345_weight_ce_multilayer_small_radius_2.pth

Type SAVE to save the model, or ABORT to cancel:  SAVE


✅ Model saved to ../../../../../projects/academic/sangst/zahidhas/Model_Weights/Monet/Monet_hyper_sarkar_head_64_Derm12345_weight_ce_multilayer_small_radius_2.pth


In [12]:
# loading block
PATH = '../../../../../projects/academic/sangst/zahidhas/Model_Weights/Monet/Monet_hyper_sarkar_head_64_Derm12345_weight_ce_multilayer_small_radius_2.pth'

response = input(
    f"\nCheckpoint path:\n{PATH}\n\n"
    "Type LOAD to load the model, or ABORT to cancel: "
).strip()

if response == "LOAD":

    checkpoint = torch.load(PATH, map_location=device)

    msg = model.load_state_dict(
        checkpoint["model_state_dict"],
        strict=False)

    print("Missing:", msg.missing_keys)
    print("Unexpected:", msg.unexpected_keys)

    # Load optimizer only if you want to RESUME training
    load_optimizer = input(
        "\nType OPTIMIZER to also load optimizer state, "
        "or SKIP to keep the current optimizer: ").strip()

    if load_optimizer == "OPTIMIZER":
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        print("✅ Optimizer state loaded.")
    else:
        print("⏭️ Optimizer state not loaded.")

    # scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    start_epoch = checkpoint["epoch"]

    print(f"✅ Model loaded from {PATH}")
    print(f"Checkpoint epoch: {start_epoch}")

elif response == "ABORT":
    print("🛑 Load aborted.")

else:
    print(f"❌ Invalid response '{response}'. Nothing was loaded.")


Checkpoint path:
../../../../../projects/academic/sangst/zahidhas/Model_Weights/Monet/Monet_hyper_sarkar_head_64_Derm12345_weight_ce_multilayer_small_radius_2.pth

Type LOAD to load the model, or ABORT to cancel:  LOAD


/scratch/26547344/ipykernel_704171/141202555.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(PATH, map_location=device)


Missing: []
Unexpected: []



Type OPTIMIZER to also load optimizer state, or SKIP to keep the current optimizer:  OPTIMIZER


✅ Optimizer state loaded.
✅ Model loaded from ../../../../../projects/academic/sangst/zahidhas/Model_Weights/Monet/Monet_hyper_sarkar_head_64_Derm12345_weight_ce_multilayer_small_radius_2.pth
Checkpoint epoch: 4


In [12]:
model.logit_scale = nn.Parameter(torch.tensor(2.0).log(), requires_grad = False) 
model.logit_scale 

Parameter containing:
tensor(0.6931)

In [12]:
L.log_map0(model.l_target).pow(2).sum(-1).sqrt()

tensor([0.9198, 1.0170, 1.0535, 0.9294, 1.0445, 0.8936, 0.9859, 0.9998, 1.0354,
        0.9332, 0.9435, 1.0355, 1.0454, 0.9494, 0.9081, 1.0002, 1.0029, 0.9000,
        1.0673, 0.9781, 0.9043, 1.0213, 1.0317, 0.9473, 1.0148, 0.9844, 0.9203,
        1.0093, 0.8385, 0.9690, 0.9882, 0.9808, 0.9175, 0.9294, 0.9615, 1.0593,
        0.9533, 0.8686, 1.0867, 0.8922], device='cuda:0')